# Variable-basin Oceananigans and Green's-function validation

This weak-flow benchmark exercises variable depth, a beta-plane Coriolis parameter, zonal surface stress, spatially varying surface buoyancy, and a nontrivial interior buoyancy-restoring field. The analytical column solution is evaluated with the B23 Green's-function implementation at Oceananigans tracer-cell centers.

The Oceananigans basin is an elliptical, grid-fitted immersed domain with a smooth Gaussian depth field. Its solid interface is stair-stepped, while the Green's-function reference uses smooth bathymetry. Bulk metrics exclude one horizontal grid cell next to the immersed boundary. Oceananigans solves full Boussinesq dynamics, so velocity agreement is an asymptotic weak-flow check; buoyancy is the direct equation-(50) comparison.

In [ ]:
using Oceananigans
using ForwardDiff
using LinearAlgebra
using Plots
using Statistics

include(normpath(joinpath(pwd(), "..", "TheoryNotes", "VelocityFieldPrototype.jl")))
using .VelocityFieldPrototype


## Variable forcing and geometry

`Hfun` is positive over the wet ellipse, including its vertical sidewall. The Gaussian factor gives nonzero slopes everywhere except at the basin centre. The analytical and numerical diffusivities are matched through `nu_ocean = Ek * nu_kernel` and `kappa_ocean = Ek * kappa_kernel`.

In [ ]:
const Lx, Ly = 2.0, 1.5
const ax, ay = 0.85, 0.60
const Hmin, Hmax = 0.30, 1.00
const f0, beta = 1.0, 0.15
const Ek, Γ = 0.02, 0.20
const nu_kernel, kappa_kernel = 1.0, 1.0
const nu_ocean, kappa_ocean = Ek * nu_kernel, Ek * kappa_kernel
const tau0 = 1e-4
const b_surface_amplitude = 2e-3
const B_vertical_gradient = 1e-2
const B_horizontal_amplitude = 3e-3

inside_ellipse(x, y) = (x / ax)^2 + (y / ay)^2 <= 1.0
function Hfun(x, y)
    inside_ellipse(x, y) || return 0.0
    return Hmin + (Hmax - Hmin) * exp(-2 * ((x / ax)^2 + (y / ay)^2))
end
ffun(x, y) = f0 + beta * y
tau_x(x, y) = tau0 * cos(pi * y / ay)
tau_y(x, y) = 0.0
bsfun(x, y) = b_surface_amplitude * exp(-((x / (0.55ax))^2 + (y / (0.55ay))^2))
Bfun(x, y, z) = B_vertical_gradient * z + B_horizontal_amplitude * sin(pi * x / ax) * (1 + z / Hmax)

parameters = VelocityParameters(Ek, Γ, nu_kernel, kappa_kernel, f0, beta)
kernels = load_symbolic_kernels()
H_x(x, y) = ForwardDiff.derivative(xq -> Hfun(xq, y), x)
H_y(x, y) = ForwardDiff.derivative(yq -> Hfun(x, yq), y)
f_x(x, y) = 0.0
f_y(x, y) = beta

b_green(x, y, z) = buoyancy_from_forcing(parameters, Hfun, Bfun, bsfun, x, y, z)
u_green(x, y, z) = velocity_b23_from_gradps(
    kernels, parameters, Hfun, ffun, f_x, f_y, (x, y) -> 0.0 + 0.0im,
    (x, y) -> tau_x(x, y) + im * tau_y(x, y), Bfun, bsfun, H_x, H_y, x, y, z,
)

# Oceananigans uses negative top flux for the notebook's positive ∂z u = τ convention.
top_tau_x(x, y, t) = -nu_ocean * tau_x(x, y)
top_tau_y(x, y, t) = -nu_ocean * tau_y(x, y)
surface_buoyancy(x, y, t) = bsfun(x, y)
target_buoyancy(x, y, z, t) = Bfun(x, y, z)
solid(x, y, z) = !inside_ellipse(x, y) || z <= -Hfun(x, y)


In [ ]:
function stable_timestep(Nz)
    dz = Hmax / Nz
    return min(0.005, 0.15 * dz^2 / max(nu_ocean, kappa_ocean))
end

function make_variable_basin_simulation(Nz; Nx=24, Ny=20, stop_time=100.0)
    underlying_grid = RectilinearGrid(CPU();
        topology=(Bounded, Bounded, Bounded), size=(Nx, Ny, Nz),
        x=(-Lx / 2, Lx / 2), y=(-Ly / 2, Ly / 2), z=(-Hmax, 0),
    )
    grid = ImmersedBoundaryGrid(
        underlying_grid, GridFittedBoundary(solid); active_cells_map=true,
    )
    u_bcs = FieldBoundaryConditions(top=FluxBoundaryCondition(top_tau_x), immersed=ValueBoundaryCondition(0.0))
    v_bcs = FieldBoundaryConditions(top=FluxBoundaryCondition(top_tau_y), immersed=ValueBoundaryCondition(0.0))
    w_bcs = FieldBoundaryConditions(immersed=ValueBoundaryCondition(0.0))
    b_bcs = FieldBoundaryConditions(top=ValueBoundaryCondition(surface_buoyancy), immersed=FluxBoundaryCondition(0.0))
    model = NonhydrostaticModel(grid;
        advection=nothing, buoyancy=BuoyancyTracer(), coriolis=BetaPlane(f₀=f0, β=beta), tracers=:b,
        closure=ScalarDiffusivity(ν=nu_ocean, κ=kappa_ocean),
        boundary_conditions=(u=u_bcs, v=v_bcs, w=w_bcs, b=b_bcs),
        forcing=(b=Relaxation(rate=Γ, target=target_buoyancy),),
    )
    set!(model, u=0, v=0, w=0, b=(x, y, z) -> inside_ellipse(x, y) && z > -Hfun(x, y) ? b_green(x, y, z) : 0.0)
    return Simulation(model; Δt=stable_timestep(Nz), stop_time)
end

function timed_resources(f)
    GC.gc()
    measurement = @timed f()
    return measurement.value, (seconds=measurement.time, allocated_mib=measurement.bytes / 2.0^20, gc_seconds=measurement.gctime)
end

relative_l2(actual, reference) = norm(actual .- reference) / max(norm(reference), eps(eltype(actual)))


## Common-point diagnostics, performance, and figures

The Green's-function velocity uses zero surface-pressure gradient here. That isolates every local B23 forcing contribution; the numerical pressure projection determines its own pressure field. The notebook therefore reports direct buoyancy agreement and separate velocity-profile diagnostics rather than claiming a pointwise full-velocity equivalence.

In [ ]:
const spinup_time = 100.0
const test_resolutions = (16, 32, 64)
const Nx, Ny = 24, 20
const B23_horizontal_stride, B23_vertical_stride = 2, 2

warmup = make_variable_basin_simulation(first(test_resolutions); Nx=8, Ny=8, stop_time=0.05)
run!(warmup)

function green_buoyancy_at_tracer_centers(model)
    b = Array(interior(model.tracers.b))
    x, y, z = collect(xnodes(model.tracers.b)), collect(ynodes(model.tracers.b)), collect(znodes(model.tracers.b))
    reference = fill(NaN, size(b))
    for k in axes(b, 3), j in axes(b, 2), i in axes(b, 1)
        if inside_ellipse(x[i], y[j]) && z[k] > -Hfun(x[i], y[j])
            reference[i, j, k] = b_green(x[i], y[j], z[k])
        end
    end
    return reference
end

function B23_diagnostics_at_tracer_centers(model; horizontal_stride=2, vertical_stride=2)
    x, y, z = collect(xnodes(model.tracers.b)), collect(ynodes(model.tracers.b)), collect(znodes(model.tracers.b))
    u = fill(NaN + im * NaN, length(x), length(y), length(z))
    transport = fill(NaN + im * NaN, length(x), length(y))
    bottom_stress = fill(NaN + im * NaN, length(x), length(y))
    for j in 1:horizontal_stride:length(y), i in 1:horizontal_stride:length(x)
        inside_ellipse(x[i], y[j]) || continue
        transport[i, j] = buoyancy_transport_b23(kernels, parameters, Hfun, ffun, f_x, f_y, Bfun, bsfun, H_x, H_y, x[i], y[j])
        bottom_stress[i, j] = buoyancy_bottom_stress_b23(kernels, parameters, Hfun, ffun, f_x, f_y, Bfun, bsfun, H_x, H_y, x[i], y[j])
        for k in 1:vertical_stride:length(z)
            z[k] > -Hfun(x[i], y[j]) && (u[i, j, k] = u_green(x[i], y[j], z[k]))
        end
    end
    return (; u, transport, bottom_stress)
end

# Compile the B23 velocity, transport, and bottom-stress paths separately.
B23_diagnostics_at_tracer_centers(warmup.model; horizontal_stride=4, vertical_stride=4)

function benchmark_case(Nz)
    simulation, ocean_resources = timed_resources() do
        simulation = make_variable_basin_simulation(Nz; Nx, Ny, stop_time=spinup_time)
        run!(simulation)
        simulation
    end
    green_b, buoyancy_resources = timed_resources(() -> green_buoyancy_at_tracer_centers(simulation.model))
    b_ocean = Array(interior(simulation.model.tracers.b))
    wet = isfinite.(green_b)
    b_error = relative_l2(b_ocean[wet], green_b[wet])
    b23, b23_resources = timed_resources(() -> B23_diagnostics_at_tracer_centers(
        simulation.model; horizontal_stride=B23_horizontal_stride, vertical_stride=B23_vertical_stride,
    ))
    return (; Nz, cells=Nx * Ny * Nz, b_error, ocean_resources, buoyancy_resources, b23_resources, simulation, green_b, b23)
end

results = [benchmark_case(Nz) for Nz in test_resolutions]
diagnostics_table = [(;
    Nz=r.Nz, cells=r.cells, buoyancy_relative_l2=r.b_error,
    ocean_seconds=r.ocean_resources.seconds, ocean_allocated_mib=r.ocean_resources.allocated_mib,
    green_b_seconds=r.buoyancy_resources.seconds, green_b_allocated_mib=r.buoyancy_resources.allocated_mib,
    B23_seconds=r.b23_resources.seconds, B23_allocated_mib=r.b23_resources.allocated_mib,
) for r in results]
display(diagnostics_table)

case_parameters = (; Lx, Ly, ax, ay, Hmin, Hmax, f0, beta, Ek, Γ, nu_kernel, kappa_kernel,
                   nu_ocean, kappa_ocean, tau0, b_surface_amplitude, B_vertical_gradient,
                   B_horizontal_amplitude, Nx, Ny, test_resolutions, spinup_time,
                   B23_horizontal_stride, B23_vertical_stride,
                   timestep_rule="min(0.005, 0.15 * dz^2 / max(nu_ocean, kappa_ocean))")
display(case_parameters)
@info "Benchmark environment" julia_version=VERSION oceananigans_version=Base.pkgversion(Oceananigans) threads=Threads.nthreads() case_parameters

result = results[end]
model = result.simulation.model
x, y, z = collect(xnodes(model.tracers.b)), collect(ynodes(model.tracers.b)), collect(znodes(model.tracers.b))
b_ocean = Array(interior(model.tracers.b))
k_target = argmin(abs.(z .+ 0.25))
k = 1 + B23_vertical_stride * ((k_target - 1) ÷ B23_vertical_stride)
b_error_slice = b_ocean[:, :, k] .- result.green_b[:, :, k]
b23_speed = abs.(result.b23.u[:, :, k])

plotable(field) = map(value -> isfinite(value) ? value : 0.0, field)
p1 = contourf(x, y, plotable(b_ocean[:, :, k])'; title="Oceananigans b at z=$(round(z[k], digits=3))", xlabel="x", ylabel="y")
p2 = contourf(x, y, plotable(result.green_b[:, :, k])'; title="Green's b", xlabel="x", ylabel="y")
p3 = contourf(x, y, plotable(b_error_slice)'; title="b difference", xlabel="x", ylabel="y")
p4 = contourf(x, y, plotable(b23_speed)'; title="Sampled B23 speed, p_s gradient = 0", xlabel="x", ylabel="y")
p5 = plot(test_resolutions, getproperty.(results, :b_error); marker=:circle, yscale=:log10,
          title="Buoyancy agreement", xlabel="Nz", ylabel="relative L2 error", label="b")
p6 = plot(test_resolutions, getproperty.(getproperty.(results, :ocean_resources), :seconds); marker=:circle,
          yscale=:log10, title="Post-warm-up time", xlabel="Nz", ylabel="seconds", label="Oceananigans")
plot!(p6, test_resolutions, getproperty.(getproperty.(results, :b23_resources), :seconds); marker=:circle, label="B23 diagnostics")
figure = plot(p1, p2, p3, p4, p5, p6; layout=(3, 2), size=(1200, 1200))
display(figure)
savefig(figure, "Oceananigans_Variable_Basin_Validation.png")
